# Data Generation Ausgrid
## Imports

In [2]:
import os
import pandas as pd

## Load Data
### Directories

In [3]:
base_dir = os.path.join(os.getcwd(), '..')
data_dir = os.path.join(base_dir, 'data', 'ausgrid')

### Load Data

In [8]:
csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
df_list = [pd.read_csv(os.path.join(data_dir, f), low_memory=False) for f in csv_files]

## Inspect Each Dataset

In [10]:
for name, df in zip(csv_files, df_list):
    print(f"File: {name}, Shape: {df.shape}, Columns: {df.columns.tolist()}, Missing Values: {df.isna().sum().sum()}")

File: Solar home 2010-2011.csv, Shape: (269736, 53), Columns: ['2010-2011 Solar home electricity data - Before using this data, please read the attached pdf document "Ausgrid solar home electricity data notes (Aug 2014).pdf"', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37', 'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41', 'Unnamed: 42', 'Unnamed: 43', 'Unnamed: 44', 'Unnamed: 45', 'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48', 'Unnamed: 49', 'Unnamed: 50', 'Unnamed: 51', 'Unnamed: 52'], M

The three Ausgrid files cover solar-home electricity data for 2010-2011, 2011-2012, and 2012-2013. Each file contains approximately 269,000 rows and 53-54 columns. The first row's explanatory text was read as a columns header, leaving most columns named `'Unnamed'`. The latter two files also report many missing values.

### Skip the explanatory text

In [11]:
df_list = [
    pd.read_csv(os.path.join(data_dir, f), skiprows=1, low_memory=False)
    for f in csv_files]

`skiprows=1` skips the explanatory first row, so pandas uses the next row for the column headers. This merely skips the row during import. It doesn't modify the CSV file itself. 

### Inspect the `Row Quality` Column

The `2011-2012` and `2012-2013` datasets have an additional `Row Quality` column. The Ausgrid documentation claims `Row Quality` distinguishes actual readings from non-actual readings. Inspect this column.

In [12]:
for name, df in zip(csv_files, df_list):
    if "Row Quality" in df.columns:
        print(f"\n{name}")
        print(df["Row Quality"].value_counts(dropna=False))


Solar home 2011-2012.csv
Row Quality
NaN    270304
Name: count, dtype: int64

Solar home 2012-2013.csv
Row Quality
NaN    268557
Name: count, dtype: int64


This prints the counts of each `Row Quality` value, including missing values, for datasets that contain the column. In both datasets, every `Row Quality ` value is missing. So this column provides no usable information. Delete this column.

In [13]:
df_list = [df.drop(columns="Row Quality", errors="ignore") for df in df_list]

This removes the column from each DataFrame that has it. `errors="ignore"` avoids an error for the dataset without that column.

## Merge all three years.

Stack the three yearly DataFrames into one with `concat`.

In [15]:
merged_df = pd.concat(df_list, ignore_index=True, sort=False)

Inspect the merged dataset.

In [16]:
print("Shape:", merged_df.shape)
display(merged_df.head())
merged_df.info()
display(merged_df.isna().sum().sort_values(ascending=False).head(10))

Shape: (808596, 53)


,Customer,Generator Capacity,Postcode,Consumption Category,date,0:30,1:00,1:30,2:00,2:30,...,19:30,20:00,20:30,21:00,21:30,22:00,22:30,23:00,23:30,0:00
0,1,3.78,2076,GC,1-Jul-10,0.303,0.471,0.083,0.121,0.361,...,0.495,0.54,0.406,0.543,0.495,0.216,0.378,0.128,0.078,0.125
1,1,3.78,2076,CL,1-Jul-10,1.250,1.244,1.256,0.744,0.019,...,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.075
2,1,3.78,2076,GG,1-Jul-10,0.000,0.000,0.000,0.000,0.000,...,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
3,1,3.78,2076,GC,2-Jul-10,0.116,0.346,0.122,0.079,0.120,...,0.262,1.10,1.012,0.817,0.526,0.335,0.402,0.142,0.120,0.111
4,1,3.78,2076,CL,2-Jul-10,1.238,1.238,1.256,1.250,0.169,...,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.088


<class 'pandas.DataFrame'>
RangeIndex: 808596 entries, 0 to 808595
Data columns (total 53 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Customer              808596 non-null  int64  
 1   Generator Capacity    808596 non-null  float64
 2   Postcode              808596 non-null  int64  
 3   Consumption Category  808596 non-null  str    
 4   date                  808596 non-null  str    
 5   0:30                  808596 non-null  float64
 6   1:00                  808596 non-null  float64
 7   1:30                  808596 non-null  float64
 8   2:00                  808596 non-null  float64
 9   2:30                  808596 non-null  float64
 10  3:00                  808596 non-null  float64
 11  3:30                  808596 non-null  float64
 12  4:00                  808596 non-null  float64
 13  4:30                  808596 non-null  float64
 14  5:00                  808596 non-null  float64
 15  5:30       

Customer                0
Generator Capacity      0
Postcode                0
Consumption Category    0
date                    0
0:30                    0
1:00                    0
1:30                    0
2:00                    0
2:30                    0
dtype: int64

The merged dataset has 808,596 rows and 53 columns. It contains customer and locations details, a data and consumption category, and 48 half-hourly readings. All columns have a value in every row, and the ten columns with the most missing values each have zero missing values. Note that `date` is currently stored as text, not as a datetime.

### Convert wide format to long format

In [18]:
# Half-hour columns
time_cols = merged_df.columns[5:]

df_long = merged_df.melt(
    id_vars=df.columns[:5],
    value_vars=time_cols,
    var_name='time',
    value_name='kWh')

This reshapes the data from wide to the tidier long format. It keeps the first five columns as identifying information, and turns the 48 half-hour columns into two columns: `time` and `kWh`. The result, `df_long`, has one row per customer/date/category/time reading.

In [19]:
display(df_long.head(10))

,Customer,Generator Capacity,Postcode,Consumption Category,date,time,kWh
0,1,3.78,2076,GC,1-Jul-10,0:30,0.303
1,1,3.78,2076,CL,1-Jul-10,0:30,1.250
2,1,3.78,2076,GG,1-Jul-10,0:30,0.000
3,1,3.78,2076,GC,2-Jul-10,0:30,0.116
4,1,3.78,2076,CL,2-Jul-10,0:30,1.238
5,1,3.78,2076,GG,2-Jul-10,0:30,0.000
6,1,3.78,2076,GC,3-Jul-10,0:30,0.730
7,1,3.78,2076,CL,3-Jul-10,0:30,1.250
8,1,3.78,2076,GG,3-Jul-10,0:30,0.000
9,1,3.78,2076,GC,4-Jul-10,0:30,0.155


## Convert datetime columns

The dataset uses two text columns for `date` and `time`. Convert this to a single standard datetime column with the proper variable type. Using a datetime type allows pandas to sort, filter, and perform calculations on datetime values.

In [20]:
datetime = pd.to_datetime(
    df_long['date'].astype(str) + " " + df_long['time'],
    format='mixed',
    dayfirst=True,
    errors='coerce')

This combines each row's `date` and `time` strings, then converts them into a pandas datetime value. `format=mixed` lets pandas infer the format separately for each value. `dayfirst=True` interprets dates as day-month-year. `errors='coerce'` turns any invalid or unparseable date/time values in `NaT` (missing datetime) instead of raising an error. Next, check for missing datetimes.

In [21]:
print(f"Invalid datetimes: {datetime.isna().sum()}/{len(datetime)}") 

Invalid datetimes: 0/38812608


There are no missing values, meaning the entire column was converted sucessfully. Next, add the converted `datetime` column to the original dataframe, and remove the old `date` and `time` columns.

In [22]:
df_long['datetime'] = datetime
df_long.drop(columns=['date', 'time'], inplace=True)

## Create load columns

The `Consumption Category` column contains three unique values: `'CL'` for controlled load, `'GC'` for general consumption, and `'GG'` for gross generation. The `kWh` columns contains the corresponding energy value. We will use this information to construction three new columns: `'Pload'` for total consumption, `'Pgen'` for generation, and `'Pnet'` for net consumption. 

### Create pivot table

In [24]:
df_loadconverted = df_long.pivot_table(
    index=['Customer', 'Generator Capacity', 'Postcode', 'datetime'],
    columns='Consumption Category',
    values='kWh',
    aggfunc='sum').reset_index()

This reshapes `df_long` so each row represents one customer at one datetime, with separate columns for each consumption category. If multiple readings share the same customer, datetime, and category, `aggfunc='sum'` adds them together. If each combination is unique, the sum leaves the reading unchanges. `reset_index()` returns the identifying fields to regular columns. Now, the table has three new columns: `'CL'`, `'GC'`, and `'GG'`.

### Create load columns

In [31]:
df_loadconverted['Pload'] = df_loadconverted['CL'] + df_loadconverted['GC']
df_loadconverted['Pgen'] = df_loadconverted['GG']
df_loadconverted['Pnet'] = df_loadconverted['Pload'] + df_loadconverted['Pgen']

KeyError: 'CL'

Remove the original consumption category columns.

In [ ]:
df_loadconverted = df_loadconverted.drop(columns=["CL", "GC", "GG"], errors="ignore")

Inspect the resulting dataframe.

In [ ]:
print(f"Shape: {df_loadconverted.shape}")
print(f"Columns: {df_loadconverted.columns.tolist()}")
print(f"Missing values: {df_loadconverted.isna().sum().sum()}")
display(df_loadconverted.head(10))

Shape: (15778512, 7)
Columns: ['Customer', 'Generator Capacity', 'Postcode', 'datetime', 'Pload', 'Pgen', 'Pnet']
Missing values: 17045856


Consumption Category,Customer,Generator Capacity,Postcode,datetime,Pload,Pgen,Pnet
0,1,3.78,2076,2010-07-01 00:00:00,1.200,0.0,1.200
1,1,3.78,2076,2010-07-01 00:30:00,1.553,0.0,1.553
2,1,3.78,2076,2010-07-01 01:00:00,1.715,0.0,1.715
3,1,3.78,2076,2010-07-01 01:30:00,1.339,0.0,1.339
4,1,3.78,2076,2010-07-01 02:00:00,0.865,0.0,0.865
5,1,3.78,2076,2010-07-01 02:30:00,0.380,0.0,0.380
6,1,3.78,2076,2010-07-01 03:00:00,1.329,0.0,1.329
7,1,3.78,2076,2010-07-01 03:30:00,0.191,0.0,0.191
8,1,3.78,2076,2010-07-01 04:00:00,0.098,0.0,0.098
9,1,3.78,2076,2010-07-01 04:30:00,0.104,0.0,0.104


The new dataset has 15,778,512 rows and 7 columns: customer and location details, datetime, and `Pload`, `Pgen`, and `Pnet`. It contains 17,045,856 missing cells in total, meaning values must be missing from multiple columns. Inspect which columns are missing values, and how many are missing from each:

In [32]:
display(df_loadconverted.isna().sum().sort_values(ascending=False))

Consumption Category
Pload                 8522928
Pnet                  8522928
Customer                    0
Postcode                    0
Generator Capacity          0
datetime                    0
Pgen                        0
dtype: int64

Load information is missing for some datetimes, but generation data is available for all datetimes. Generation data alone may be useful for forecasting, so we don't want to get rid of these records. So, we leave in the explicit missing values for now.